In [3]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

DATA_FILE = "Water Quality.csv"
LOCATION_COLUMN = "MonitoringLocationID"
PARAMETER_COLUMN = "CharacteristicName"
VALUE_COLUMN = "ResultValue"
UNIT_COLUMN = "ResultUnit"
OUTPUT_DIR = "task1_outputs"
MIN_GROUP_SIZE = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)
df = pd.read_csv(DATA_FILE, low_memory=False)

required = [LOCATION_COLUMN, PARAMETER_COLUMN, VALUE_COLUMN]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}\nAvailable columns: {list(df.columns)}")

# Convert measurements to numeric; non-numeric results become missing.
df[VALUE_COLUMN] = pd.to_numeric(df[VALUE_COLUMN], errors="coerce")
df = df.dropna(subset=[LOCATION_COLUMN, PARAMETER_COLUMN, VALUE_COLUMN]).copy()

# Keep parameters with at least two observations in at least two locations.
location_counts = df.groupby([PARAMETER_COLUMN, LOCATION_COLUMN])[VALUE_COLUMN].count()
usable_parameters = []
for parameter in df[PARAMETER_COLUMN].unique():
    counts = location_counts.loc[parameter] if parameter in location_counts.index.get_level_values(0) else pd.Series(dtype=int)
    eligible = (counts >= MIN_GROUP_SIZE).sum()
    if eligible >= 2:
        usable_parameters.append(parameter)

if not usable_parameters:
    raise ValueError(
        "No parameter has at least two measurements in at least two locations. "
        "Try grouping by MonitoringLocationName or ActivityMediaName, or inspect your data."
    )

# Descriptive statistics by parameter and location.
summary = (
    df[df[PARAMETER_COLUMN].isin(usable_parameters)]
    .groupby([PARAMETER_COLUMN, LOCATION_COLUMN])[VALUE_COLUMN]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)
summary.to_csv(os.path.join(OUTPUT_DIR, "descriptive_statistics_by_location.csv"), index=False)

print("Dataset shape:", df.shape)
print("Unique locations:", df[LOCATION_COLUMN].nunique())
print("Parameters included:", len(usable_parameters))
print("\nDescriptive statistics preview:")
print(summary.head(20).round(4).to_string(index=False))

sns.set_theme(style="whitegrid")
test_results = []

# One distribution plot per parameter; if there are many locations, show the
# 8 locations with the most observations for readability.
for parameter in usable_parameters:
    part = df[df[PARAMETER_COLUMN] == parameter].copy()
    top_locations = part[LOCATION_COLUMN].value_counts().head(8).index
    plot_data = part[part[LOCATION_COLUMN].isin(top_locations)].copy()
    safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(parameter))

    plt.figure(figsize=(10, 5))
    sns.histplot(data=part, x=VALUE_COLUMN, kde=True)
    plt.title(f"Distribution of {parameter}")
    plt.xlabel(f"ResultValue ({part[UNIT_COLUMN].dropna().iloc[0] if UNIT_COLUMN in part.columns and part[UNIT_COLUMN].notna().any() else 'unit not specified'})")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"hist_{safe}.png"), dpi=250)
    plt.close()

    plt.figure(figsize=(max(8, len(top_locations) * 1.1), 5))
    sns.boxplot(data=plot_data, x=LOCATION_COLUMN, y=VALUE_COLUMN)
    plt.title(f"{parameter}: comparison across top monitoring locations")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"box_{safe}.png"), dpi=250)
    plt.close()

    # Compare the two locations with the most measurements for this parameter.
    counts = part[LOCATION_COLUMN].value_counts()
    if len(counts) >= 2:
        loc_a, loc_b = counts.index[:2]
        a = part.loc[part[LOCATION_COLUMN] == loc_a, VALUE_COLUMN].dropna()
        b = part.loc[part[LOCATION_COLUMN] == loc_b, VALUE_COLUMN].dropna()
        if len(a) >= MIN_GROUP_SIZE and len(b) >= MIN_GROUP_SIZE:
            stat, p = ttest_ind(a, b, equal_var=False, nan_policy="omit")
            test_results.append({
                "Parameter": parameter,
                "Location_1": loc_a,
                "Location_2": loc_b,
                "n_1": len(a),
                "n_2": len(b),
                "mean_1": a.mean(),
                "mean_2": b.mean(),
                "welch_t_statistic": stat,
                "p_value": p
            })

if test_results:
    tests = pd.DataFrame(test_results)
    # Bonferroni correction across the parameter-level comparisons.
    tests["p_value_bonferroni"] = np.minimum(tests["p_value"] * len(tests), 1.0)
    tests["significant_after_correction"] = tests["p_value_bonferroni"] < 0.05
    tests.to_csv(os.path.join(OUTPUT_DIR, "location_comparison_t_tests.csv"), index=False)
    print("\nWelch t-test comparisons for the two most-sampled locations per parameter:")
    print(tests.round(5).to_string(index=False))
else:
    print("\nNo eligible two-location comparisons were available for t-tests.")

print(f"\nAll Task 1 outputs saved in: {OUTPUT_DIR}")
print(
    "\nInterpretation: compare the means, medians, distributions, and boxplots. "
     "A small adjusted p-value suggests a difference in group means. It does not ")

Dataset shape: (229916, 40)
Unique locations: 58
Parameters included: 60

Descriptive statistics preview:
CharacteristicName MonitoringLocationID  count    mean  median     std   min   max
 Alkalinity, total           NB01BR0006     27 17.5444    7.33 20.5412  1.73 75.60
 Alkalinity, total           NB01BR0008     35 32.1780   26.80 22.1644  2.90 72.10
 Alkalinity, total           NB01BR0087     34 25.2418   20.20 17.8341  2.36 56.10
 Alkalinity, total           NB01BR0112     34 31.8285   25.65 19.1297  3.90 69.90
 Alkalinity, total           NB01BR0114     36 22.2456   17.70 13.6702  3.40 47.90
 Alkalinity, total           NS01DC0001    128 24.6165   24.05  9.3495  7.16 64.00
 Alkalinity, total           NS01DC0203     33  3.9609    3.59  1.5123  1.64  7.07
 Alkalinity, total           NS01DD0016    155 51.4852   54.00 11.6108 17.50 72.80
 Alkalinity, total           NS01DR0001     63 21.6073   22.50  6.6783  7.07 37.90
 Alkalinity, total           NS01EA0001      4  2.2150    2.05  